# Interaction features (products of features)

_Feature Engineering_

**Multiply two features so a linear model can say 'the effect of one depends on the other'.**

A linear model predicts with a weighted sum: $\hat y = w_0 + w_1 x_1 + w_2 x_2 + \dots$. Each feature gets one weight, and that weight is the feature's effect regardless of the other features. So a plain linear model is structurally unable to say "the effect of $x_1$ changes depending on $x_2$".

       An interaction feature is simply the product of two features, $x_1 x_2$, added as a new column. Now the model can learn a weight $w_{12}$ on that product. Rewriting, the effect of $x_1$ becomes $w_1 + w_{12} x_2$ — it now depends on $x_2$. That is exactly the "the effect of location depends on age" behavior we wanted, and it is an AND-like combination: the product is large only when both features are large.

---

This notebook is a practice scaffold for the **Interaction features (products of features)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## See it for yourself: the problem, then the fix

Run this cell. It plots the **raw** data and reproduces the problem it causes, then plots the **engineered** data and shows the fix — with before/after numbers.

In [ ]:
# Interaction features: adding the product x_i * x_j so a LINEAR model can
# capture that two features only matter TOGETHER (their joint sign here).
#
# THE STORY:
#   A plain linear model draws ONE straight line. If the class label depends on
#   the COMBINATION of two features (an XOR pattern), no single straight line
#   can split the data -> the model is stuck near 50% (coin-flip).
#   FIX: hand the model the product feature x1*x2. Now the same straight-line
#   model can separate the classes, because x1*x2 already encodes "same sign?".

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

# ----------------------------------------------------------------------------
# 1. LOAD DATA. There is no bundled XOR dataset, so we BUILD one with a fixed
#    rng (reproducible). Two features x1, x2 drawn from a normal distribution.
#    Rule: class y = 1 when x1 and x2 have the SAME sign (both +, or both -),
#    and y = 0 when they have OPPOSITE signs. This is the classic XOR pattern.
# ----------------------------------------------------------------------------
rng = np.random.default_rng(0)          # fixed seed -> same numbers every run
n = 400
x1 = rng.normal(size=n)                 # feature 1
x2 = rng.normal(size=n)                 # feature 2
X_raw = np.column_stack([x1, x2])       # raw feature matrix, shape (n, 2)
# "same sign" -> product is positive. That is exactly y = 1.
y = (np.sign(x1) == np.sign(x2)).astype(int)

# ----------------------------------------------------------------------------
# 2. VISUALIZE THE RAW DATA. Scatter colored by class. You can SEE the four
#    quadrants: top-right & bottom-left are one class, the other two quadrants
#    are the other class. No single straight line can carve that apart.
# ----------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.scatter(x1[y == 1], x2[y == 1], c="tab:blue",   s=18, label="y=1 (same sign)")
ax.scatter(x1[y == 0], x2[y == 0], c="tab:orange", s=18, label="y=0 (opp. sign)")
ax.axhline(0, color="gray", lw=0.8)
ax.axvline(0, color="gray", lw=0.8)
ax.set_xlabel("x1"); ax.set_ylabel("x2")
ax.set_title("RAW data: XOR pattern\n(no straight line can split the 4 quadrants)")
ax.legend(loc="upper right", fontsize=8)

# ----------------------------------------------------------------------------
# 3. REPRODUCE THE PROBLEM. Fit a LINEAR model (LogisticRegression) on the raw
#    [x1, x2] features. Because a straight line cannot separate XOR, accuracy
#    sits around 0.5 -- the model is barely better than guessing.
# ----------------------------------------------------------------------------
clf = LogisticRegression()
clf.fit(X_raw, y)
acc_raw = clf.score(X_raw, y)           # train accuracy on raw features
print(f"PROBLEM  -- linear model on raw [x1, x2]      accuracy = {acc_raw:.3f}")

# ----------------------------------------------------------------------------
# 4. APPLY THE TECHNIQUE: add the INTERACTION feature x1*x2, then visualize it.
#    Plot x1*x2 (one axis) against the class. The two classes now sit on
#    OPPOSITE sides of 0: same-sign points have x1*x2 > 0, opposite-sign < 0.
#    A single threshold at 0 splits them -> linearly separable on this axis.
# ----------------------------------------------------------------------------
inter = (x1 * x2).reshape(-1, 1)        # the new feature, shape (n, 1)
X_eng = np.column_stack([x1, x2, x1 * x2])   # raw features PLUS the product

ax = axes[1]
# small random y-jitter only so overlapping points are visible (cosmetic).
jitter = rng.normal(scale=0.04, size=n)
ax.scatter(inter[y == 1, 0], y[y == 1] + jitter[y == 1],
           c="tab:blue",   s=18, label="y=1 (same sign)")
ax.scatter(inter[y == 0, 0], y[y == 0] + jitter[y == 0],
           c="tab:orange", s=18, label="y=0 (opp. sign)")
ax.axvline(0, color="red", lw=1.2, ls="--", label="threshold at x1*x2 = 0")
ax.set_xlabel("interaction feature  x1 * x2")
ax.set_ylabel("class  y")
ax.set_title("ENGINEERED feature: x1*x2 vs class\n(classes split cleanly at 0)")
ax.legend(loc="center right", fontsize=8)

plt.tight_layout()
plt.show()

# ----------------------------------------------------------------------------
# 5. SHOW THE FIX. Re-fit the SAME LogisticRegression on [x1, x2, x1*x2].
#    The product feature does the heavy lifting -> accuracy jumps to ~1.0.
# ----------------------------------------------------------------------------
clf2 = LogisticRegression()
clf2.fit(X_eng, y)
acc_eng = clf2.score(X_eng, y)
print(f"FIX      -- same model + interaction x1*x2    accuracy = {acc_eng:.3f}")

# Explicit before/after summary.
print(f"PROBLEM (raw): {acc_raw:.3f}   ->   FIX (engineered): {acc_eng:.3f}")

## Reference implementation — scikit-learn

In [ ]:
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

# Online News Popularity (UCI) -- from the book's GitHub:
#   github.com/alicezheng/feature-engineering-book  ->  data/OnlineNewsPopularity.csv
df = pd.read_csv("OnlineNewsPopularity.csv")
df.columns = [c.strip() for c in df.columns]   # the CSV has leading spaces in headers

# target = number of shares; drop the non-predictive columns
y = df["shares"].values
X = df.drop(columns=["url", "timedelta", "shares"]).values
print("base feature matrix:", X.shape)         # ~ (39644, 58)

# --- add ALL pairwise interaction features (degree-2, no squared terms) ---
pf = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X2 = pf.fit_transform(X)
print("with interactions:  ", X2.shape)        # ~ (39644, 1711) -- quadratic blow-up

# --- compare cross-validated R^2: base features vs base + interactions ---
lr = LinearRegression()
r2_base  = cross_val_score(lr, X,  y, cv=5, scoring="r2")
r2_inter = cross_val_score(lr, X2, y, cv=5, scoring="r2")

print("R^2 base features:        %.4f  (+/- %.4f)" % (r2_base.mean(),  r2_base.std()))
print("R^2 base + interactions:  %.4f  (+/- %.4f)" % (r2_inter.mean(), r2_inter.std()))
# The book observes a small improvement from the interaction features --
# bought at the cost of ~30x more columns and longer training time.

## Visualize the data & results

_On a real dataset, does adding pairwise interaction features lift a linear model's accuracy — and what does it cost in feature count?_

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline

Xall, y = load_breast_cancer(return_X_y=True)  # 569 samples, 30 features
X = Xall[:, :5]                                 # the first 5 'mean' features
print("base:", X.shape)                         # (569, 5)

# add all pairwise interaction features (no squared terms)
pf = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X2 = pf.fit_transform(X)
print("with interactions:", X2.shape)           # (569, 15) = 5 + C(5,2)=10

# standardize first, then fit -- products are well-scaled this way
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
acc_base  = cross_val_score(clf, X,  y, cv=5, scoring="accuracy").mean()
acc_inter = cross_val_score(clf, X2, y, cv=5, scoring="accuracy").mean()
print("acc base:               %.4f" % acc_base)    # ~0.9244
print("acc base+interactions:  %.4f" % acc_inter)   # ~0.9279  (small real lift)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You fit a plain LinearRegression to predict house price from two standardized features, location_score and buyer_age, and the model underfits: it predicts the same location premium for every buyer, but you know young and older buyers value neighborhoods differently. What single new feature lets a linear model express this, and what does its coefficient mean?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Name what the plain model can and cannot do. — _$\hat y = w_0 + w_1\,\text{loc} + w_2\,\text{age}$ gives location one fixed weight $w_1$ — the same premium for every age. It structurally cannot make the location effect vary with age._
- Form the interaction. — _Add the product column $\text{loc}\times\text{age}$. The model becomes $\hat y = w_0 + w_1\text{loc} + w_2\text{age} + w_{12}(\text{loc}\cdot\text{age})$._
- Read the new marginal effect. — _$\partial\hat y/\partial\,\text{loc} = w_1 + w_{12}\,\text{age}$ — the location premium now slides with buyer age, exactly the behavior you wanted._

**Answer:** Add the interaction feature $\text{location\_score} \times \text{buyer\_age}$ as a new column (e.g. via PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)). The linear model can then learn a weight $w_{12}$ on it, and the effect of location becomes $w_1 + w_{12}\cdot\text{buyer\_age}$ — no longer constant, but sliding with age. The coefficient $w_{12}$ says how much the location premium changes per unit of (standardized) buyer age: positive means older buyers value good locations more; negative means less. This is Zheng & Casari's "the effect of location depends on age" example, encoded as one product column. Because both features were standardized first, the product is well-scaled.

</details>

**Problem 2.** Your feature matrix has $d = 50$ numeric columns. You run PolynomialFeatures(degree=2, interaction_only=True, include_bias=False). How many columns come out, how many would the full polynomial give, and what two precautions does the book recommend?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Count the interaction-only output. — _Interaction-only keeps the $d$ originals and adds one product per unordered pair: $\binom{50}{2}=\frac{50\cdot 49}{2}=1225$. Total $= 50 + 1225 = 1275$._
- Count the full polynomial. — _The full degree-2 expansion also adds the $d=50$ squared terms $x_i^2$: total $= 50 + 1225 + 50 = 1325$ (1275 features plus 50 squares)._
- Recall the trade-off. — _Both blow up as $O(d^2)$, so many columns risk overfitting and slow training — the book pairs interactions with regularization and feature selection._

**Answer:** Interaction-only: $50 + \binom{50}{2} = 50 + 1225 = \mathbf{1275}$ columns. Full polynomial (interaction_only=False): it also adds the 50 squared terms $x_i^2$, giving $1275 + 50 = \mathbf{1325}$. Both grow like $O(d^2)$, so 50 features became well over a thousand. The book's two precautions: (1) use regularization (ridge or lasso) so the extra columns cannot memorize noise, and (2) follow with feature selection to keep only the interactions that actually help. It also helps to standardize first so the products are well-scaled.

</details>

**Problem 3.** A colleague adds all pairwise interaction features to a random forest and to a logistic regression, expecting both to improve. One barely changes and may even get slightly worse; the other can improve. Which is which, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what each model can already represent. — _Trees split sequentially: a split on $x_1$ followed by a split on $x_2$ already encodes their interaction, so the forest learns interactions natively._
- Recall the linear model's limit. — _Logistic (a linear-in-parameters model) sums each feature times one weight and cannot represent a product unless you hand it the product column._
- Predict the outcome. — _Interaction features mostly add redundant, noisy columns to the forest (no gain, extra cost) but give the logistic model genuinely new expressive power._

**Answer:** The logistic regression can improve; the random forest barely changes (or slightly worsens). A linear/logistic model adds each feature times one weight and is structurally blind to feature products, so handing it $x_i x_j$ columns genuinely expands what it can fit. A random forest already learns interactions by splitting on one feature and then another, so the hand-built products are mostly redundant — they add dimensionality, training cost, and a little noise without new information, which can even nudge validation accuracy down. This is exactly the book's guidance: reserve interaction features for models that cannot learn interactions themselves (linear/logistic), not for trees or neural nets.

</details>